In [1]:
# ================================================================
# NOTEBOOK 1 — SETUP + DATASET (CELEBA-HQ 256 → 128)
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install einops kornia lpips matplotlib scikit-image tqdm --quiet

import os, torch, shutil
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ================================================================
# DATASET PATH SU DRIVE
# ================================================================
root_drive = "/content/drive/MyDrive/Colab Notebooks/celeba_hq_raw/celeba_hq_256"

# ================================================================
# FUNZIONE ROBUSTA CHE EVITA OSERROR (Drive I/O errors)
# ================================================================
def safe_listdir(path):
    files = []
    for f in os.scandir(path):
        try:
            if f.name.lower().endswith(".jpg"):
                files.append(f.name)
        except:
            continue
    return sorted(files)

# ================================================================
# OPZIONALE: CREA SOTTOINSIEME LOCALE SU /content
# ================================================================
local_root = "/content/celeba_hq_256_subset"
MAX_IMAGES = 8000  # puoi ridurre se vuoi

if not os.path.exists(local_root):
    os.makedirs(local_root, exist_ok=True)
    files = safe_listdir(root_drive)
    files = files[:MAX_IMAGES]
    print(f"Copia di {len(files)} immagini in {local_root}...")
    for f in files:
        src = os.path.join(root_drive, f)
        dst = os.path.join(local_root, f)
        shutil.copy(src, dst)
else:
    print("Subset locale già presente:", local_root)

root = local_root  # usa il subset locale

# ================================================================
# DATASET CLASS
# ================================================================
class CelebAHQ(Dataset):
    def __init__(self, root, size=128):
        self.root = root
        self.files = safe_listdir(root)
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),  # [0,1] → [-1,1]
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.files[idx])).convert("RGB")
        return self.transform(img)

# ================================================================
# DATALOADER
# ================================================================
dataset = CelebAHQ(root)
loader  = DataLoader(dataset, batch_size=8, shuffle=True,
                     num_workers=2, pin_memory=True)

print("Dataset size:", len(dataset))


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 101.5 MB/s eta 0:00:00
Device: cuda
Copia di 8000 immagini in /content/celeba_hq_256_subset...
Dataset size: 8000


In [2]:
# ================================================================
# NOTEBOOK 2 — TRAINING DECOLOR COLD DIFFUSION
# ================================================================

import torch, torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# Riusa 'device', 'loader' e 'dataset' dal Notebook 1

# ================================================================
# UNet COMPATTO CON CONDITIONING SUL TEMPO (γ = t / (T-1))
# ================================================================
class Block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, ch=48, in_ch=4):   # 3 canali RGB + 1 canale tempo
        super().__init__()
        self.d1   = Block(in_ch, ch)
        self.pool = nn.MaxPool2d(2)
        self.d2   = Block(ch, ch*2)

        self.u1   = Block(ch*2 + ch, ch)
        self.final = nn.Conv2d(ch, 3, 1)

    def forward(self, x, gamma):
        """
        x:   [B, 3, H, W]  immagine degradata
        gamma: [B, 1, 1, 1] tempo normalizzato in [0,1]
        """
        B, C, H, W = x.shape
        gamma_map = gamma.expand(-1, 1, H, W)   # broadcasting
        inp = torch.cat([x, gamma_map], dim=1)  # [B, 4, H, W]

        x1 = self.d1(inp)
        x2 = self.d2(self.pool(x1))
        x2_up = F.interpolate(x2, scale_factor=2, mode='bilinear', align_corners=False)

        xu = torch.cat([x2_up, x1], dim=1)
        out = self.final(self.u1(xu))
        return out

model = UNet().to(device)

# ================================================================
# DECOLOR FORWARD PROCESS (DETERMINISTICO, MENO DISTRUTTIVO)
# ================================================================
T = 50                # numero di step di degradazione
alpha_max = 0.7       # quanta parte di grayscale al massimo (t = T-1)

def decolor_forward(x, t, T=T, alpha_max=alpha_max):
    """
    x    : [B, 3, H, W]           in [-1, 1]
    t    : [B,1,1,1] o scalare    in [0, T-1]
    T    : numero di step totali
    """
    # t può essere float o long
    t = t.float()
    gamma = t / (T - 1)           # in [0,1]
    alpha = alpha_max * gamma     # in [0, alpha_max]

    gray = x.mean(1, keepdim=True).repeat(1, 3, 1, 1)
    return (1 - alpha) * x + alpha * gray

# ================================================================
# LOSS E OPTIMIZER
# ================================================================
def L1(a, b):
    return (a - b).abs().mean()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
scaler = torch.cuda.amp.GradScaler()

# ================================================================
# TRAINING — RIDOTTO MA EFFICIENTE
# ================================================================
n_steps = 4000   # invece di 12000
step    = 0

model.train()
pbar = tqdm(total=n_steps)

while step < n_steps:
    for x in loader:
        x = x.to(device)

        # campiona t da 0..T-1 (include anche t=0)
        t = torch.randint(0, T, (x.size(0), 1, 1, 1), device=device)
        x_t = decolor_forward(x, t)          # immagine degradata
        gamma = t.float() / (T - 1)          # normalizzato

        with torch.cuda.amp.autocast():
            pred = model(x_t, gamma)         # predizione x0
            loss = L1(pred, x)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        step += 1
        pbar.update(1)

        if step % 200 == 0:
            pbar.set_description(f"Step {step}/{n_steps} - Loss {loss.item():.4f}")

        if step >= n_steps:
            break

pbar.close()

# (poi salva il modello)
save_path = "/content/drive/MyDrive/decolor_model_cold_diff.pt"
torch.save(model.state_dict(), save_path)
print("Modello salvato in:", save_path)


/tmp/ipython-input-2690002936.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  0%|          | 0/4000 [00:00<?, ?it/s]/tmp/ipython-input-2690002936.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Step 4000/4000 - Loss 0.0139: 100%|██████████| 4000/4000 [01:39<00:00, 40.11it/s]

Modello salvato in: /content/drive/MyDrive/decolor_model_cold_diff.pt


In [3]:
# ================================================================
# NOTEBOOK 3 — RICOSTRUZIONE + METRICHE
# ================================================================

import os, torch
from torchvision.utils import save_image
from tqdm import tqdm
from math import log10
import lpips
from skimage.metrics import structural_similarity as ssim
import numpy as np

# Riusa: device, loader, dataset, decolor_forward, T, alpha_max, UNet

# ================================================================
# CARICA MODELLO ALLENATO
# ================================================================
model = UNet().to(device)
load_path = "/content/drive/MyDrive/decolor_model_cold_diff.pt"
model.load_state_dict(torch.load(load_path, map_location=device))
model.eval()
print(f"Modello caricato da: {load_path}")

T = 50

# ================================================================
# ALGORITMO 1 — ONE-STEP RECON
# ================================================================
@torch.no_grad()
def alg1(x):
    """
    x : [B,3,H,W] immagine pulita (solo per valutazione).
    Ritorna:
      x_T  : immagine dopo T-1 step DeColor
      rec1 : ricostruzione da x_T con un solo passo
    """
    B = x.size(0)
    t_T = torch.full((B,1,1,1), T-1, device=device)
    x_T = decolor_forward(x, t_T)

    gamma_T = t_T.float() / (T - 1)
    rec1 = model(x_T, gamma_T)
    return x_T, rec1

# ================================================================
# ALGORITMO 2 — RESIDUAL CORRECTION (COLD DIFFUSION)
# ================================================================
@torch.no_grad()
def alg2_from_xT(x_T):
    """
    x_T : stato al tempo T-1 (degradato)
    Implementa:
      x_{t-1} = x_t - D( x0_hat, t ) + D( x0_hat, t-1 )
    """
    x_t = x_T.clone()

    for t in reversed(range(T)):
        B = x_t.size(0)
        t_tensor = torch.full((B,1,1,1), t, device=device)
        gamma_t = t_tensor.float() / (T - 1)

        # stima x0
        x0_hat = model(x_t, gamma_t)

        if t > 0:
            x_t_minus1 = x_t - decolor_forward(x0_hat, t_tensor) \
                             + decolor_forward(x0_hat, t_tensor - 1)
        else:
            # ultimo step: usiamo direttamente x0_hat
            x_t_minus1 = x0_hat

        x_t = x_t_minus1

    return x_t   # stimatore finale di x0

# ================================================================
# CARTELLE PER SALVARE ALCUNE IMMAGINI
# ================================================================
os.makedirs("orig", exist_ok=True)
os.makedirs("decolor", exist_ok=True)
os.makedirs("rec_alg1", exist_ok=True)
os.makedirs("rec_alg2", exist_ok=True)

print("Cartelle create: orig/, decolor/, rec_alg1/, rec_alg2/")

# ================================================================
# METRICHE: PSNR, SSIM, LPIPS
# ================================================================
loss_fn_alex = lpips.LPIPS(net='alex').to(device)
COMPUTE_LPIPS = False  # metti True solo per il run finale

def psnr(img1, img2):
    """
    img1, img2 in [0,1], tensor [3,H,W]
    """
    mse = torch.mean((img1 - img2) ** 2).item()
    if mse == 0:
        return 100.0
    return 10 * log10(1.0 / mse)

# ================================================================
# VALUTAZIONE SU N_IMMAGINI (ridotto)
# ================================================================
N_EVAL   = 80    # ridotto per velocizzare
N_SAVE   = 32    # quante salvare su disco per la tesi
counter  = 0

psnr_list  = []
ssim_list  = []
lpips_list = []

model.eval()

pbar = tqdm(loader)
for batch in pbar:
    x = batch.to(device)   # immagini pulite [-1,1]

    # Alg.1: x_T e una ricostruzione one-step
    x_T, rec1 = alg1(x)

    # Alg.2: ricostruzione iterativa da x_T
    rec2 = alg2_from_xT(x_T)

    B = x.size(0)
    for i in range(B):
        if counter >= N_EVAL:
            break

        # Scala a [0,1] per PSNR/SSIM/salvataggio
        x_clean = (x[i].detach().cpu()   + 1) / 2
        x_rec2  = (rec2[i].detach().cpu() + 1) / 2
        x_decol = (x_T[i].detach().cpu() + 1) / 2
        x_rec1  = (rec1[i].detach().cpu() + 1) / 2

        # === METRICHE ===
        # PSNR
        psnr_val = psnr(x_clean, x_rec2)
        psnr_list.append(psnr_val)

        # SSIM (skimage vuole HWC, [0,1])
        clean_np = x_clean.numpy().transpose(1,2,0)
        rec_np   = x_rec2.numpy().transpose(1,2,0)
        ssim_val = ssim(clean_np, rec_np, data_range=1.0, channel_axis=-1)
        ssim_list.append(ssim_val)

        # LPIPS (usa tensori [-1,1])
        if COMPUTE_LPIPS:
            x_clean_11 = x[i].unsqueeze(0)      # [1,3,H,W]
            x_rec2_11  = rec2[i].unsqueeze(0)
            lp_val = loss_fn_alex(x_clean_11, x_rec2_11).item()
            lpips_list.append(lp_val)

        # === SALVATAGGIO IMMAGINI PER TESI (primi N_SAVE) ===
        if counter < N_SAVE:
            save_image(x_clean, f"orig/{counter}.png" )
            save_image(x_decol, f"decolor/{counter}.png")
            save_image(x_rec1,  f"rec_alg1/{counter}.png")
            save_image(x_rec2,  f"rec_alg2/{counter}.png")

        counter += 1

    pbar.set_description(f"Eval: {counter}/{N_EVAL} imgs")

    if counter >= N_EVAL:
        break

# ================================================================
# RISULTATI MEDI
# ================================================================
psnr_mean  = float(np.mean(psnr_list))
ssim_mean  = float(np.mean(ssim_list))
lpips_mean = float(np.mean(lpips_list)) if len(lpips_list) > 0 else None

print("\n=== METRICHE SU", N_EVAL, "IMMAGINI (Alg.2) ===")
print("PSNR :", psnr_mean)
print("SSIM :", ssim_mean)
if lpips_mean is not None:
    print("LPIPS:", lpips_mean)
else:
    print("LPIPS non calcolato (COMPUTE_LPIPS = False)")
print("\nImmagini di esempio salvate in orig/, decolor/, rec_alg1/, rec_alg2/")


Modello caricato da: /content/drive/MyDrive/decolor_model_cold_diff.pt
Cartelle create: orig/, decolor/, rec_alg1/, rec_alg2/
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 150MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


Eval: 80/80 imgs:   1%|          | 9/1000 [00:08<16:08,  1.02it/s]


=== METRICHE SU 80 IMMAGINI (Alg.2) ===
PSNR : 38.012468934897306
SSIM : 0.9859779477119446
LPIPS non calcolato (COMPUTE_LPIPS = False)

Immagini di esempio salvate in orig/, decolor/, rec_alg1/, rec_alg2/


In [ ]:
import torch
from torchvision.utils import save_image

model.eval()

# Prendiamo 1 immagine dal dataset
x0 = next(iter(loader))[0:1].to(device)   # [1,3,H,W]

# Tempo massimo t = T-1 (T=50)
t = torch.full((1,1,1,1), T-1, device=device)

# Degradazione deterministica finale
x_T = decolor_forward(x0, t)

# Ricostruzione Alg. 2
xrec = alg2_from_xT(x_T)

# Denormalizza per salvataggio
x0_d   = (x0   *0.5 + 0.5).clamp(0,1)
xT_d   = (x_T  *0.5 + 0.5).clamp(0,1)
xrec_d = (xrec *0.5 + 0.5).clamp(0,1)

# Salva i tre file
save_image(x0_d,   "/content/decolor_original.png")
save_image(xT_d,   "/content/decolor_degraded.png")
save_image(xrec_d, "/content/decolor_alg2.png")

print("✔ Immagini DECOLOR salvate!")


In [ ]:
from PIL import Image

orig = Image.open("/content/decolor_original.png").resize((256,256))
deg  = Image.open("/content/decolor_degraded.png").resize((256,256))
rec  = Image.open("/content/decolor_alg2.png").resize((256,256))

W, H = orig.width * 3, orig.height
combined = Image.new("RGB", (W, H))

combined.paste(orig, (0,0))
combined.paste(deg,  (256,0))
combined.paste(rec,  (512,0))

combined.save("/content/decolor_row_clean.png")
combined
